# Pairs Trading Research — Walkthrough

This notebook is a thin demo of the `pairs` package. Every analysis here is reproduced inside the Streamlit app and locked in by `tests/test_pipeline.py`.

For the full reasoning behind each design choice, see the linked docs in `docs/`. For the headline result and the resume bullet, see the project `README.md`.

In [ ]:
# Make src/ importable when running from notebooks/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from pairs.data import PAIR_REGISTRY, load_pair
from pairs.signals import static_hedge_ratio
from pairs.pipeline import run_pipeline, PipelineParams
from statsmodels.tsa.stattools import adfuller

print("Available pairs:")
for pid, p in PAIR_REGISTRY.items():
    print(f"  {pid:<12} {p.label}")

## 1. Load a pair and look at the prices

In [ ]:
leg1, leg2, pair = load_pair("HDFC_ICICI")
print(f"{pair.label}: {len(leg1)} trading days, {leg1.index[0].date()} - {leg1.index[-1].date()}")

fig, ax1 = plt.subplots(figsize=(13, 4.5))
ax1.plot(leg1.index, leg1.values, color="#1f77b4", linewidth=1.4, label=pair.leg1)
ax1.set_ylabel(f"{pair.leg1} (Rs)", color="#1f77b4")
ax2 = ax1.twinx()
ax2.plot(leg2.index, leg2.values, color="#ff7f0e", linewidth=1.4, label=pair.leg2)
ax2.set_ylabel(f"{pair.leg2} (Rs)", color="#ff7f0e")
ax1.set_title(f"{pair.label} - daily closing prices")
plt.tight_layout()
plt.show()

## 2. Fit the static hedge and check cointegration

The strategy assumes the spread `leg1 - beta * leg2` is stationary. We fit beta by OLS, build the spread, and run an ADF test on it. A p-value below 0.05 says the spread mean-reverts.

In [ ]:
beta = static_hedge_ratio(leg1, leg2)
spread = leg1 - beta * leg2

cutoff = pd.Timestamp("2026-01-30")
adf_full   = adfuller(spread.dropna())
adf_stable = adfuller(spread.loc[spread.index <= cutoff].dropna())

print(f"Hedge ratio beta : {beta:.4f}")
print(f"ADF on full sample   : p = {adf_full[1]:.4f}  ({'cointegrated' if adf_full[1] < 0.05 else 'NOT cointegrated'})")
print(f"ADF on stable regime : p = {adf_stable[1]:.4f}  ({'cointegrated' if adf_stable[1] < 0.05 else 'NOT cointegrated'})")
print()
print("The p-value rises an order of magnitude once the post-Feb 2026 period")
print("is included - that is the HDFC governance event weakening cointegration.")

## 3. Run the full pipeline on the stable regime

This is the headline result that goes on the resume.

In [ ]:
result = run_pipeline("HDFC_ICICI", PipelineParams(end="2026-01-30"))
s = result["summary"]

print(f"  Period            : Apr 2024 - Jan 2026")
print(f"  Trades            : {s['n_trades']}")
print(f"  Win rate          : {s['win_rate_pct']:.1f}%")
print(f"  Total P&L         : Rs {s['total_pnl']:.2f}")
print(f"  Sharpe Ratio      : {s['sharpe']:.3f}")
print(f"  Max drawdown      : Rs {s['max_drawdown']:.2f}  ({s['max_drawdown_pct']:.1f}% of peak)")
print(f"  Profit factor     : {s['profit_factor']:.2f}x")

## 4. Plot the cumulative P&L and position over time

In [ ]:
daily = result["daily"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(daily.index, daily["cum_pnl"], color="#2ca02c", linewidth=1.5)
ax1.fill_between(daily.index, daily["cum_pnl"], 0,
                  where=daily["cum_pnl"] >= 0, alpha=0.15, color="#2ca02c")
ax1.fill_between(daily.index, daily["cum_pnl"], 0,
                  where=daily["cum_pnl"] < 0, alpha=0.15, color="#d62728")
ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax1.set_ylabel("Cumulative P&L (Rs)")
ax1.set_title(f"HDFC/ICICI - cumulative P&L (Sharpe {s['sharpe']:.2f})")

ax2.fill_between(daily.index, daily["position"], where=daily["position"] > 0,
                  alpha=0.5, color="#2ca02c", label="Long spread")
ax2.fill_between(daily.index, daily["position"], where=daily["position"] < 0,
                  alpha=0.5, color="#d62728", label="Short spread")
ax2.set_yticks([-1, 0, 1])
ax2.set_yticklabels(["Short", "Flat", "Long"])
ax2.set_ylabel("Position")
ax2.legend(loc="upper right", fontsize=9)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax2.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Multi-pair generalisation test

Run the same pipeline on every registered pair, with no parameter tuning per pair. The Augmented Dickey-Fuller p-value (run pre-trade) should predict which pairs the strategy will work on.

In [ ]:
rows = []
for pid, p in PAIR_REGISTRY.items():
    l1, l2, _ = load_pair(pid)
    b = static_hedge_ratio(l1, l2)
    sp = l1 - b * l2
    adf_p = adfuller(sp.dropna())[1]
    r = run_pipeline(pid, PipelineParams(end="2026-01-30"))["summary"]
    rows.append({
        "pair": p.label,
        "beta": round(b, 3),
        "ADF p": round(adf_p, 4),
        "trades": r["n_trades"],
        "win %": round(r["win_rate_pct"], 1),
        "Sharpe": round(r["sharpe"], 2),
        "P&L (Rs)": round(r["total_pnl"], 2),
        "verdict": "cointegrated" if adf_p < 0.05 else "NOT cointegrated",
    })
pd.DataFrame(rows)

## What's next

- The Streamlit app (`app/streamlit_app.py`) lets you tune every parameter live and see the strategy update.
- `docs/rolling_hedge_analysis.md` walks through why static beta beats rolling beta on this 2-year dataset.
- `docs/parameter_sensitivity.md` shows the (entry x window) Sharpe heatmap and explains why our chosen point isn't the highest cell.
- `docs/multi_pair_analysis.md` explains the ADF gating story above.
- `docs/interview_qa.md` has the predictable interview questions with model answers.